In [0]:
dbutils.fs.ls("/Workspace/Shared/Data/")

In [0]:
data_json=spark.read.format("json").load("dbfs:/FileStore/orders_product.json")

In [0]:
display(data_json)

In [0]:
data=[
  {
    "order_id": 1001,
    "customer_id": 501,
    "order_date": "2026-01-10",
    "status": "completed",
    "total_amount": 250.75,
    "shipping_address": {
      "city": "New York",
      "state": "NY",
      "zipcode": "10001"
    },
    "products": [
      { "product_id": 1, "name": "Laptop", "price": 200.50, "quantity": 1 },
      { "product_id": 2, "name": "Mouse", "price": 25.25, "quantity": 2 }
    ]
  },
  {
    "order_id": 1002,
    "customer_id": 502,
    "order_date": "2026-01-11",
    "status": "pending",
    "total_amount": 120.00,
    "shipping_address": {
      "city": "Los Angeles",
      "state": "CA",
      "zipcode": "90001"
    },
    "products": [
      { "product_id": 3, "name": "Keyboard", "price": 60.00, "quantity": 2 }
    ]
  },
  {
    "order_id": 1003,
    "customer_id": 503,
    "order_date": "2026-01-11",
    "status": "completed",
    "total_amount": 75.00,
    "shipping_address": {
      "city": "Chicago",
      "state": "IL",
      "zipcode": "60601"
    },
    "products": [
      { "product_id": 4, "name": "Headphones", "price": 75.00, "quantity": 1 }
    ]
  },
  {
    "order_id": 1004,
    "customer_id": 504,
    "order_date": "2026-01-12",
    "status": "cancelled",
    "total_amount": 300.00,
    "shipping_address": {
      "city": "Houston",
      "state": "TX",
      "zipcode": "77001"
    },
    "products": [
      { "product_id": 5, "name": "Monitor", "price": 300.00, "quantity": 1 }
    ]
  },
  {
    "order_id": 1005,
    "customer_id": 505,
    "order_date": "2026-01-12",
    "status": "completed",
    "total_amount": 180.00,
    "shipping_address": {
      "city": "Phoenix",
      "state": "AZ",
      "zipcode": "85001"
    },
    "products": [
      { "product_id": 6, "name": "Webcam", "price": 90.00, "quantity": 2 }
    ]
  },
  {
    "order_id": 1006,
    "customer_id": 506,
    "order_date": "2026-01-13",
    "status": "completed",
    "total_amount": 45.00,
    "shipping_address": {
      "city": "Philadelphia",
      "state": "PA",
      "zipcode": "19019"
    },
    "products": [
      { "product_id": 7, "name": "USB Cable", "price": 15.00, "quantity": 3 }
    ]
  },
  {
    "order_id": 1007,
    "customer_id": 507,
    "order_date": "2026-01-13",
    "status": "pending",
    "total_amount": 500.00,
    "shipping_address": {
      "city": "San Antonio",
      "state": "TX",
      "zipcode": "78201"
    },
    "products": [
      { "product_id": 8, "name": "Tablet", "price": 500.00, "quantity": 1 }
    ]
  },
  {
    "order_id": 1008,
    "customer_id": 508,
    "order_date": "2026-01-14",
    "status": "completed",
    "total_amount": 95.50,
    "shipping_address": {
      "city": "San Diego",
      "state": "CA",
      "zipcode": "92101"
    },
    "products": [
      { "product_id": 9, "name": "Power Bank", "price": 47.75, "quantity": 2 }
    ]
  },
  {
    "order_id": 1009,
    "customer_id": 509,
    "order_date": "2026-01-14",
    "status": "completed",
    "total_amount": 60.00,
    "shipping_address": {
      "city": "Dallas",
      "state": "TX",
      "zipcode": "75201"
    },
    "products": [
      { "product_id": 10, "name": "Charger", "price": 30.00, "quantity": 2 }
    ]
  },
  {
    "order_id": 1010,
    "customer_id": 510,
    "order_date": "2026-01-15",
    "status": "pending",
    "total_amount": 210.00,
    "shipping_address": {
      "city": "San Jose",
      "state": "CA",
      "zipcode": "95101"
    },
    "products": [
      { "product_id": 11, "name": "Smart Speaker", "price": 105.00, "quantity": 2 }
    ]
  }
]


In [0]:
display(data)

In [0]:
df_json=spark.createDataFrame(data)
display(df_json)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_flat = df_json.withColumn("products", explode("products"))\
    .select(
        "customer_id", "order_date", "order_id",
        col("products.product_id").alias("prod_id"),
        col("products.name").alias("name"),
        col("products.price").alias("price"),
        col("products.quantity").alias("quantity"),
        col("shipping_address.city").alias("city"),
        col("shipping_address.state").alias("state"),
        col("shipping_address.zipcode").alias("zipcode"),
        "status", "total_amount"
    )

In [0]:
display(df_flat)

In [0]:
df_flat=df_flat.withColumn("currency_type",lit("USD"))
df_flat=df_flat.withColumn("price",col("price").cast("double"))

In [0]:
df_auto=spark.sparkContext.parallize(data).toDF()
df1=flatten_df(df_auto)
display(df1)

In [0]:
data_csv=[{
    "Currency":"USD",
    "Exchange_rate":1.3
},
    {
    "Currency":"CAD",
    "Exchange_rate":0.7
    }      ]

In [0]:
df_csv=spark.createDataFrame(data_csv)
display(df_csv)

In [0]:
df_join_all=df_flat.join(df_csv,df_flat.currency_type == df_csv.Currency,how="left")

In [0]:
display(df_join_all)

In [0]:
df2=df_join_all.withColumn("price_in_cad",col("price")*1.3).withColumn("total_in_cad",col("total_amount")*1.3)

In [0]:
display(df2)

In [0]:
import random

In [0]:
df3 = df2.withColumn("delivered_date", lit([f"01-{random.randint(1,30)}-2026" for _ in range(df2.count())][0]))

In [0]:
display(df3)

In [0]:
df3=df3.withColumn("delivered_date",date_format(to_date(col("delivered_date"),"MM-dd-yyyy"),"yyyy-MM-dd"))

In [0]:
df3=df3.withColumn("delivered_date",to_date(col("delivered_date"),"yyyy-MM-dd"))

In [0]:
df3.printSchema()

In [0]:
df3=df3.withColumn("order_date",to_date(col("order_date"),"yyyy-MM-dd"))

In [0]:
display(df3)

In [0]:
df3=df3.withColumn("current_timing",current_timestamp())

In [0]:
display(df3)

In [0]:
df3=df3.withColumn("current_timing1",to_utc_timestamp(col("current_timing"),"Asia/Kolkata"))

In [0]:
display(df3)

In [0]:
%sql
create catalog windows;
create schema windows.practice;

In [0]:
%sql
create view windows.practice.sales as 
select * from values
(1, 'A', '2024-01-01', 100),
(2, 'A', '2024-01-02', 150),
(3, 'A', '2024-01-03', 200),
(4, 'B', '2024-01-01', 80),
(5, 'B', '2024-01-02', 120),
(6, 'B', '2024-01-03', 160)
as sales(id,name,sale_date,amount)

In [0]:
%sql
--Previous day sales per department
select *, LAG(amount,1,0) over( partition by name order by sale_date) as previous_day_sale from windows.practice.sales

In [0]:
%sql
-- Next day sales per department
select *, LEAD(amount,1,0) over(partition by 'name' order by 'sale_date') as next_day_sales from windows.practice.sales 

In [0]:
%sql
-- Running total of amount per department
select *, sum(amount) over(partition by name order by sale_date rows between unbounded preceding and current row) as running_total from windows.practice.sales

In [0]:
%sql
-- First sale amount per department
select *, first_value(amount) over(partition by name order by sale_date) as first_sale from windows.practice.sales

In [0]:
%sql
-- Last sale amount per department

select *,  last_value(amount) over(partition by name order by sale_date rows between unbounded preceding and unbounded following) as last_sale from windows.practice.sales

In [0]:
%sql
-- Difference between today’s sales and previous day
select *, amount - lag(amount,1,0) over( order by sale_date) as sale_dif_prev from windows.practice.sales

## Joins

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
customers=spark.createDataFrame([    (1, "Alice", "Canada"),
    (2, "Bob", "USA"),
    (3, "Charlie", "UK"),
    (4, "David", "India")],['id','name','country'])
display(customers)    

In [0]:
orders=spark.createDataFrame([    (101, 1, 500),
    (102, 1, 300),
    (103, 2, 700),
    (104, 5, 400)],['order_id','cust_id','amount'])
display(orders)    

In [0]:
#inner-join
df_inner_join=customers.join(orders,customers.id==orders.cust_id,how='inner').select('id','name','country','order_id','amount')
display(df_inner_join)

In [0]:
df_left=customers.join(orders,customers.id==orders.cust_id,how='left')
display(df_left)

In [0]:
# right outer
df_join_right=customers.join(orders, customers.id==orders.cust_id,how='right')
df_join_right.display()

In [0]:
# full outer
df_join_outer=customers.join(orders,  customers.id==orders.cust_id, how='outer')
display(df_join_outer)

In [0]:
# left-anti
df_join_left_anti=customers.join(orders, customers.id==orders.cust_id,how='left_anti')
display(df_join_left_anti)

In [0]:
# left-semi

df_join_left_semi=customers.join(orders, customers.id==orders.cust_id,how='left_semi')
display(df_join_left_semi)

In [0]:
# cross join
df_join_cross=customers.crossJoin(orders)
display(df_join_cross)

In [0]:
# Data modelling

df_main=spark.read.format('csv').option('header','true').\
    option("inferSchema",'true').load('/Volumes/windows/practice/modelling_main/pratice_table.csv')

In [0]:
display(df_main)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_main=df_main.withColumn('order_date',to_date(col('order_date'),'dd-MM-yyyy'))
display(df_main)

In [0]:
df_main=df_main.withColumn("total_amount",col("quantity")*col("unit_price"))

In [0]:
from pyspark.sql.window import Window

In [0]:
dim_customer=df_main.select("customer_id","customer_name","customer_city").dropDuplicates(["customer_id"]).withColumn("custmer_sk",row_number().over(Window.orderBy("customer_id")))
display(dim_customer)

In [0]:
dim_product=df_main.select("product_id","product_category","product_name").dropDuplicates(['product_id']).withColumn("product_sk",row_number().over(Window.orderBy("product_id")))
display(dim_product)

In [0]:
fact_main=df_main.alias("m").join(dim_product.alias("p"),col("m.product_id")==col("p.product_id"),how='left').join(dim_customer.alias("c"),col("m.customer_id")==col("c.customer_id"),how='left').select(
    col("m.order_id"),col("m.order_date"),col("m.quantity"),col("m.unit_price"),
    col("c.custmer_sk"),col("p.product_sk"),col("m.total_amount")
)

In [0]:
display(fact_main)

In [0]:
fact_main.write.format("delta").mode("append").saveAsTable("windows.practice.fact_main")

In [0]:
%sql
describe history windows.practice.fact_main

In [0]:
display(fact_main)

In [0]:
%sql
select * from windows.practice.fact_main  version as of 0

In [0]:
%sql
restore windows.practice.fact_main to version as of 0